# 07 · Tables

This notebook writes the six tables of the paper: Table 1, Table 2 and Supplementary Tables 1–4.

* **Table 1** describes the cohort. It is computed here from the cohort, the clinical context of each ECG and the repeat-ECG sets of the
  test split (notebook 02).
* **Table 2** and **Supplementary Tables 2–4** are read from the outcome tables in `SUMMARY` (notebooks 04–06), and **Supplementary
  Table 1** combines a description of each model with its training record in `TRAINING` (notebook 03).

Each table goes to `TABLES` as markdown and as CSV, under the names `Table1`, `Table2` and `SuppTable1` … `SuppTable4`. The tables
contain aggregate numbers only. Numbers are rounded half away from zero, and 95% confidence intervals are on the second line of a cell.

The whole notebook runs in a few seconds on a CPU.

In [ ]:
%run ./00_setup_and_settings.ipynb

## Settings and shared definitions

Rows of every model table follow `MODEL_KEYS` and are grouped under the foundation model. `OVERWRITE = False` keeps tables whose files
exist in `TABLES`; set it to `True` to write every table again.

In [ ]:
import json

OVERWRITE = False

GROUP_LABEL = {"Lima": "Public model", "ECGFounder": "ECGFounder", "ECG-FM": "ECG-FM", "HuBERT-ECG": "HuBERT-ECG"}   # bold group rows
ROW_LABEL = {"public": "Lima et al. model", "frozen": "Frozen", "mimic": "Fine-tuned (MIMIC)", "code15": "Fine-tuned (CODE-15%)"}
DASH = "—"


def grouped_rows(keys, row, group_label=GROUP_LABEL):
    """Table rows of the models in `keys` (in this order), each foundation model under a bold group label; row(key) gives the cells."""
    out, last = [], None
    for key in keys:
        fm = MODELS.fm[key]
        if fm != last:
            out.append(group_label[fm])
            last = fm
        out.append([ROW_LABEL[MODELS.adapt[key]]] + row(key))
    return out


def show_table(name):
    """Print a written table (aggregate numbers only)."""
    print((TABLES / f"{name}.md").read_text(encoding="utf-8"))

## Table 1

**Table 1 | Characteristics of the ECGs, patients and repeat ECG sets.** The repeat ECG sets and the mortality analysis are from the test
split, and the mortality analysis comprises the first ECG of each patient that was followed by another ECG within 48 hours. Ages and care
settings of the repeat sets refer to the earlier ECG of each patient's pair with the shortest interval. ECG-level characteristics and
intervals are weighted so that each patient contributes equally. IQR, interquartile range; SD, standard deviation.

How the columns are computed:

* *All included ECGs* and *Test split*: every included ECG (of the test split). Age and care setting are ECG-level characteristics; each
  ECG is weighted by 1 / (number of ECGs of its patient), and the weighted SD of age is multiplied by √(n / (n − 1)), n being the number of
  ECGs. Women and death within 1 year are counted per patient, at the patient's first ECG.
* *Within 1 h, same visit* and *Within 48 h*: the repeat-ECG sets. The interval is the patient-weighted median and IQR over all pairs (each
  pair weighted by 1 / number of pairs of its patient). Age, sex, care setting and death refer to the earlier ECG of each patient's pair
  with the shortest interval (of several pairs with the same interval, the one whose earlier ECG was recorded first); each patient counts
  once.
* *Mortality analysis*: the first ECG of each patient of the mortality cohort (notebook 02).

In [ ]:
SETTING_LABEL = {"ED": "Emergency department", "ward": "Ward", "ICU": "Intensive care unit", "outpatient": "Outpatient"}
SETTINGS_ROWS = list(SETTING_LABEL.values()) + ["Other"]          # every other care setting (and an unknown one) counts as "Other"

cohort = pd.read_parquet(COHORT, columns=["idx", "subject_id", "ecg_time", "include", "split"])
cohort = cohort[cohort["include"]].set_index("idx")
cohort["time"] = pd.to_datetime(cohort["ecg_time"])
context = pd.read_parquet(ECG_CONTEXT, columns=["idx", "loc", "sex", "age", "death_1y"]).set_index("idx")


def setting_shares(loc, weights):
    """Weighted share (%) of each care setting."""
    label = pd.Series(np.asarray(loc, dtype=object)).map(lambda x: SETTING_LABEL.get(x, "Other")).to_numpy()
    total = pd.Series(np.asarray(weights, float)).groupby(label).sum()
    return {s: float(100 * total.get(s, 0.0) / total.sum()) for s in SETTINGS_ROWS}


def ecg_column(ecgs):
    """Characteristics of a set of included ECGs (one row per ECG, index idx)."""
    ctx = context.reindex(ecgs.index)
    w = patient_weights(ecgs["subject_id"])
    age = ctx["age"].to_numpy(float)
    mean = np.average(age, weights=w)
    first = ecgs.sort_values("time", kind="stable").drop_duplicates("subject_id").index      # first ECG of each patient
    return {"ecgs": len(ecgs), "patients": ecgs["subject_id"].nunique(),
            "two_or_more_pct": 100 * float((ecgs.groupby("subject_id").size() >= 2).mean()),
            "age": (mean, float(np.sqrt(np.average((age - mean) ** 2, weights=w) * len(w) / (len(w) - 1)))),
            "women_pct": 100 * float((context.loc[first, "sex"] == "F").mean()),
            "settings": setting_shares(ctx["loc"], w),
            "death_pct": 100 * float(context.loc[first, "death_1y"].mean())}


def pair_column(pairs):
    """Characteristics of a repeat-ECG set (idx_a earlier, idx_b later, subject_id, hours_apart)."""
    pairs = pairs.reset_index(drop=True)
    w = (1.0 / pairs.groupby("subject_id")["subject_id"].transform("size")).to_numpy()
    hours = pairs["hours_apart"].to_numpy(float)
    order = np.argsort(hours)
    cum = np.cumsum(w[order]) / w.sum()

    def quantile(q):
        """Patient-weighted quantile of the intervals."""
        return float(hours[order][np.searchsorted(cum, q)])

    pairs["time_a"] = cohort["time"].reindex(pairs["idx_a"]).to_numpy()
    shortest = pairs.sort_values(["subject_id", "hours_apart", "time_a"], kind="stable").drop_duplicates("subject_id")
    a = context.reindex(shortest["idx_a"].to_numpy())                          # earlier ECG of each patient's pair with the shortest interval
    return {"pairs": len(pairs), "patients": pairs["subject_id"].nunique(),
            "interval_hours": (quantile(0.5), quantile(0.25), quantile(0.75)), "age": (float(a["age"].mean()), float(a["age"].std(ddof=1))),
            "women_pct": 100 * float((a["sex"] == "F").mean()), "settings": setting_shares(a["loc"], np.ones(len(a))),
            "death_pct": 100 * float(a["death_1y"].mean())}


def mortality_column(cohort_file):
    first = context.reindex(pd.read_parquet(cohort_file)["idx_first"].to_numpy())   # first ECG of each patient of the mortality cohort
    age, died = first["age"].to_numpy(float), first["death_1y"].astype(bool).to_numpy(int)
    return {"patients": len(first), "deaths": int(died.sum()), "age": (float(age.mean()), float(age.std(ddof=1))),
            "women_pct": 100 * float((first["sex"] == "F").mean()), "death_pct": 100 * float(died.mean())}


A = ecg_column(cohort)
T = ecg_column(cohort[cohort["split"] == "test"])
S1 = pair_column(pd.read_parquet(PAIRS_1H))
S48 = pair_column(pd.read_parquet(PAIRS_48H))
M = mortality_column(MORTALITY_COHORT)
print(f"included ECGs {A['ecgs']:,} of {A['patients']:,} patients; test split {T['ecgs']:,} of {T['patients']:,}; "
      f"pairs within 1 h {S1['pairs']:,} ({S1['patients']:,} patients), within 48 h {S48['pairs']:,} ({S48['patients']:,} patients); "
      f"mortality cohort {M['patients']:,} patients, {M['deaths']:,} deaths")

In [ ]:
f, n = fmt_num, masked          # counts from 1 to 9 would be shown as "<10" (credentialed data)


def mean_sd(c):
    return f"{f(c['age'][0])} ({f(c['age'][1])})"


def interval(c, factor, unit, nd):
    q = [x * factor for x in c["interval_hours"]]
    return f"{f(q[0], nd)} ({f(q[1], nd)}–{f(q[2], nd)}) {unit}"


rows = [["ECGs, *n*", n(A["ecgs"]), n(T["ecgs"]), DASH, DASH, DASH],
        ["Pairs of ECGs, *n*", DASH, DASH, n(S1["pairs"]), n(S48["pairs"]), DASH],
        ["Patients, *n*", n(A["patients"]), n(T["patients"]), n(S1["patients"]), n(S48["patients"]), n(M["patients"])],
        ["Patients with two or more ECGs, %", f(A["two_or_more_pct"]), f(T["two_or_more_pct"]), DASH, DASH, DASH],
        ["Interval between the two ECGs, median (IQR)", DASH, DASH, interval(S1, 60, "min", 0), interval(S48, 1, "h", 1), DASH],
        ["Age, years, mean (SD)"] + [mean_sd(c) for c in (A, T, S1, S48, M)],
        ["Women, %"] + [f(c["women_pct"]) for c in (A, T, S1, S48, M)],
        "Care setting at the ECG, %"]
rows += [[f"  {s}"] + [f(c["settings"][s]) for c in (A, T, S1, S48)] + [DASH] for s in SETTINGS_ROWS]
rows += [["Death within 1 year, %"] + [f(c["death_pct"]) for c in (A, T, S1, S48, M)]]
write_table("Table1", ["Characteristic", "All included ECGs", "Test split", "Within 1 h, same visit", "Within 48 h", "Mortality analysis"],
            rows, overwrite=OVERWRITE)
show_table("Table1")

## Table 2

**Table 2 | Accuracy and test–retest reliability of AI-ECG age in the test split.** Values are for repeat ECGs of the same patient within
48 hours unless stated otherwise, with 95% confidence intervals from patient bootstrap on the second line. The MAE weights each patient
equally. The SEM is the standard deviation of the within-pair differences divided by √2. The between-patient SD is the standard deviation
of the age gaps without retest variation, derived as SEM × √(ICC/(1 − ICC)). The ICC of the age gap adjusted for age and sex is the ICC of
the residuals of the age gap after regression on age and sex. Pairs are weighted so that each patient contributes equally. Frozen, frozen
embedding with ridge regression; ICC, intraclass correlation coefficient; MAE, mean absolute error; MIMIC, MIMIC-IV-ECG; SD, standard
deviation; SEM, standard error of measurement.

In [ ]:
accuracy = pd.read_csv(SUMMARY / "accuracy.csv").set_index("key")
reliability = pd.read_csv(SUMMARY / "reliability.csv").set_index(["key", "set"])


def with_ci(r, col, nd):
    """Cell with the estimate `col` of the result row r and its 95% CI (columns col_lo, col_hi)."""
    return fmt_cell(r[col], r[f"{col}_lo"], r[f"{col}_hi"], nd)


def table2_row(key):
    r48, r1 = reliability.loc[(key, "within_48h")], reliability.loc[(key, "within_1h")]
    return [f(accuracy.loc[key, "mae_mimic_test"], 2), with_ci(r48, "median_abs_d", 2), f(r48["sem"], 1), f(r1["sem"], 1),
            f(r48["between_sd"], 1), with_ci(r48, "icc_gap", 3), with_ci(r48, "icc_gap_adj", 3)]


write_table("Table2", ["Model", "MAE (years)", "Median absolute difference within 48 h (years)", "SEM within 48 h (years)",
                       "SEM within 1 h (years)", "Between-patient SD of the age gap (years)", "ICC of the age gap",
                       "ICC of the age gap adjusted for age and sex"],
            grouped_rows(MODEL_KEYS, table2_row), overwrite=OVERWRITE)
show_table("Table2")

## Supplementary Table 1

**Supplementary Table 1 | Models and their adaptation.** Architecture and pretraining data are as reported by the developers, and numbers
of parameters were counted from the loaded networks. The fine-tuned versions were trained as described in Methods, with gradient clipping
at 1 and bfloat16 precision, and the checkpoint with the lowest validation mean absolute error, weighting each patient equally, was kept.
The selected setting is the ridge penalty, or the optimization step of the checkpoint kept (with its epoch) and the step at which training
ended.

The descriptions of the networks (first columns) are fixed text; the training settings and the selected setting are read from the
training record of each model (notebook 03).

In [ ]:
DESCRIPTION = {   # foundation model: (architecture and size, pretraining data, input)
    "ECGFounder": ("Net1D, a RegNet-based one-dimensional convolutional network; 30.8 million parameters",
                   "Supervised pretraining on 150 diagnostic labels in 10.8 million ECGs of the Harvard–Emory ECG Database; MIMIC-IV-ECG, "
                   "CODE-15% and PTB-XL not used",
                   "12 leads, 500 Hz, 10 s, standardized over the record"),
    "ECG-FM": ("wav2vec 2.0-type: convolutional encoder and 12-layer transformer (768 dimensions); 90.4 million parameters",
               "Self-supervised pretraining on 1.4 million 5-s segments from MIMIC-IV-ECG (developers' training partition) and the PhysioNet "
               "2021 challenge data, which include PTB-XL",
               "Two 5-s halves, 500 Hz, standardized per lead"),
    "HuBERT-ECG": ("HuBERT: convolutional feature extractor and 16-layer transformer (960 dimensions); 187.6 million parameters",
                   "Self-supervised pretraining on 9.1 million ECGs from 11 datasets, including MIMIC-IV-ECG, PTB-XL and CODE (except its "
                   "public test set)",
                   "Two 5-s windows, band-pass 0.05–47 Hz, scaled to [−1, 1], 100 Hz")}
LIMA_ROW = ["Lima et al. model", "One-dimensional residual convolutional network; 6.9 million parameters",
            "Supervised training for age on ECGs of 85% of the 1,558,415 patients of the CODE study; CODE-15% patients held out",
            f"12 leads, 400 Hz, 4,096 samples; amplitude ×{CODE_AMPLITUDE_RATIO:.3f}", "None (published weights)", DASH]
training = {key: json.loads((TRAINING / f"{key}.json").read_text(encoding="utf-8")) for key in MODEL_KEYS if MODELS.adapt[key] != "public"}


def power_of_ten(x):
    """1e-4 -> '1 × 10^−4^' (the carets mark a superscript in the markdown table)."""
    e = int(np.floor(np.log10(x) + 1e-9))
    return f"{round(x / 10 ** e, 6):g} × 10^{'−' if e < 0 else ''}{abs(e)}^"


def training_settings(cfg):
    batch = f"Batch {cfg['bs']}" + (f" with {cfg['accum']} accumulation steps" if cfg["accum"] > 1 else "")
    return (f"{batch}; learning rate {power_of_ten(cfg['lr_bb'])} (network), {power_of_ten(cfg['lr_head'])} (heads); "
            f"weight decay {power_of_ten(cfg['wd'])}")


def selected_setting(rec):
    if "ridge_alpha" in rec:
        return f"Ridge penalty {rec['ridge_alpha']:g}"
    epoch = rec["best_epoch"] + 1                                               # the record counts epochs from 0
    end = "early stop" if rec["early_stop"] else f"all {rec['cfg']['epochs']} epochs"
    return f"Step {rec['best_step']:,} (epoch {epoch}); training ended at step {rec['total_steps']:,} ({end})"


def table_s1_row(key):
    fm, adapt, rec = MODELS.fm[key], MODELS.adapt[key], training[key]
    if adapt == "frozen":
        return list(DESCRIPTION[fm]) + ["Ridge regression on the MIMIC-IV-ECG training split", selected_setting(rec)]
    if adapt == "mimic":
        return ["", "", "", training_settings(rec["cfg"]), selected_setting(rec)]
    same = training_settings(rec["cfg"]) == training_settings(training[key.replace("code15", "mimic")]["cfg"])
    return ["", "", "", "As above, on CODE-15%" if same else training_settings(rec["cfg"]) + ", on CODE-15%", selected_setting(rec)]


rows = ["Public model", LIMA_ROW] + grouped_rows([k for k in MODEL_KEYS if MODELS.adapt[k] != "public"], table_s1_row,
                                                 {**GROUP_LABEL, "HuBERT-ECG": "HuBERT-ECG large"})
write_table("SuppTable1", ["Model", "Architecture and size", "Pretraining or training data", "Input", "Adaptation", "Selected setting"],
            rows, overwrite=OVERWRITE)
show_table("SuppTable1")

## Supplementary Table 2

**Supplementary Table 2 | Accuracy of AI-ECG age.** Mean absolute error of AI-ECG age against chronological age, in years, for ECGs
recorded at ages of 18–88 years, each patient weighted equally. Dashes mark combinations that were not evaluated. The CODE-15% test split
was formed in this study by splitting patients 80/10/10, and PTB-XL was used only for evaluation. <sup>a</sup>PTB-XL was used to choose the
input amplitude scale of the Lima et al. model and is therefore not an independent evaluation of this model.

In [ ]:
def table_s2_row(key):
    a = accuracy.loc[key]
    cells = [f(a["mae_mimic_test"], 2), f(a["mae_code15_test"], 2), f(a["mae_ptbxl"], 2)]
    if MODELS.adapt[key] == "public":                     # Lima et al. model: footnote a on PTB-XL
        cells[2] += "^a^"
    return cells


write_table("SuppTable2", ["Model", "MIMIC-IV-ECG test split", "CODE-15% test split", "PTB-XL"], grouped_rows(MODEL_KEYS, table_s2_row),
            overwrite=OVERWRITE)
show_table("SuppTable2")

## Supplementary Table 3

**Supplementary Table 3 | Predicted sex between repeat ECGs.** Area under the receiver operating characteristic curve (AUC) for sex in the
test split, and the share of pairs of ECGs of the same patient in which the predicted sex (the sign of the logit of the sex output)
differed, within 1 hour at the same visit and within 48 hours. The Lima et al. model has no sex output and is not shown.

In [ ]:
sex = pd.read_csv(SUMMARY / "sex.csv").set_index("key")
write_table("SuppTable3", ["Model", "AUC for sex", "Predicted sex differed within 1 h, same visit (%)", "Predicted sex differed within 48 h (%)"],
            grouped_rows([k for k in MODEL_KEYS if k in sex.index],
                         lambda k: [f(sex.loc[k, "auc"], 3), f(sex.loc[k, "flip_within_1h_pct"], 1), f(sex.loc[k, "flip_within_48h_pct"], 1)]),
            overwrite=OVERWRITE)
show_table("SuppTable3")

## Supplementary Table 4

**Supplementary Table 4 | Association between the age gap and death within 1 year, with and without retest variation.** For each test
patient, the first ECG followed by another ECG within 48 hours (32,127 patients, of whom 3,944 died within 1 year, the same for every
model). Odds ratios (OR) per 10-year larger age gap of the first ECG are adjusted for age, sex and the number of ECGs within 48 hours
(2, 3, 4–5 or 6 or more). Without the number of ECGs, the ORs differed by at most 0.003 and the association hidden by at most 1.0
percentage point. The OR without retest variation was obtained by simulation–extrapolation (SIMEX), and the association hidden is 1
minus the observed logarithm of the OR divided by that from SIMEX. The OR ratio compares the mean age gap of the first five ECGs within
7 days with the age gap of the first ECG, in the patients with five or more ECGs within 7 days, adjusted for age, sex and the number of
ECGs within 7 days, with 95% confidence intervals.

In [ ]:
mortality = pd.read_csv(SUMMARY / "mortality.csv").set_index("key")


def table_s4_row(key):
    r = mortality.loc[key]
    return [with_ci(r, "or_obs", 3), with_ci(r, "or_simex", 3), with_ci(r, "hidden_pct", 0), with_ci(r, "ratio_7d", 3)]


write_table("SuppTable4", ["Model", "OR per 10-year larger age gap, observed",
                           "OR per 10-year larger age gap without retest variation (SIMEX)", "Association hidden by retest variation (%)",
                           "OR ratio, mean of five ECGs within 7 days to the first ECG"],
            grouped_rows(MODEL_KEYS, table_s4_row), overwrite=OVERWRITE)
show_table("SuppTable4")

# The legend compares these models with the same models without the number of ECGs (age and sex only; point estimates).
d = mortality.loc[MODEL_KEYS]
print(f"without the number of ECGs: largest difference of the observed OR {(d['or_obs'] - d['or_obs_age_sex']).abs().max():.4f}, "
      f"of the SIMEX OR {(d['or_simex'] - d['or_simex_age_sex']).abs().max():.4f}, "
      f"of the association hidden {(d['hidden_pct'] - d['hidden_pct_age_sex']).abs().max():.2f} percentage points")